In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parents[2]

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print(PROJECT_ROOT)

/Users/rawls/quant-lab


In [2]:
import pandas as pd
import numpy as np

from src.data.market_configs import MARKET_CONFIGS
from src.data.loader import download_market_data
from src.pipelines.strategy_returns import build_strategy_return_stack

from src.utils.metrics import (
    sharpe_ratio,
    max_drawdown,
    annualized_return,
)

from src.analysis.turnover import (
    compute_turnover,
    rolling_turnover,
    summarize_turnover,
    build_turnover_report,
)

In [3]:
#Deployment candidates

market_specs = {
    "India": {
        "config": MARKET_CONFIGS["india"],
        "signals": [
            "mr_ret_10",
            "low_vol_20",
            "mr_lowvol_blend",
        ],
    },
    "Brazil": {
        "config": MARKET_CONFIGS["brazil"],
        "signals": [
            "mom_blend",
            "mr_lowvol_blend",
        ],
    },
    "Japan": {
        "config": MARKET_CONFIGS["japan"],
        "signals": [
            "mr_ret_5",
            "mr_ret_20",
        ],
    },
}

In [4]:
from src.analysis.robustness import run_market_robustness

rebalance_tc_results = []

for market_name, spec in market_specs.items():
    spec = dict(spec)
    spec["name"] = market_name

    result = run_market_robustness(
        market_spec=spec,
        target_vol=0.10,
        rebalance_frequencies=[1, 2, 5, 10, "weekly"],
        transaction_costs=[0, 2, 5, 10, 20, 50],
    )

    rebalance_tc_results.append(result)

rebalance_tc_results_df = (
    pd.concat(rebalance_tc_results, ignore_index=True)
    .round(3)
    .sort_values(["Market", "Rebalance Frequency", "Cost bps"])
)

rebalance_tc_results_df

,Market,Signal Combo,Rebalance Frequency,Cost bps,Sharpe,MDD,CAGR,Calmar,Mean Turnover,Median Turnover,P95 Turnover,% Trading Days
30,Brazil,mom_blend + mr_lowvol_blend,1,0,1.662,-0.259,0.161,0.620,0.512,0.5,0.80,99.635
31,Brazil,mom_blend + mr_lowvol_blend,1,2,1.383,-0.288,0.131,0.456,0.512,0.5,0.80,99.635
32,Brazil,mom_blend + mr_lowvol_blend,1,5,0.963,-0.347,0.088,0.254,0.512,0.5,0.80,99.635
33,Brazil,mom_blend + mr_lowvol_blend,1,10,0.264,-0.467,0.020,0.044,0.512,0.5,0.80,99.635
34,Brazil,mom_blend + mr_lowvol_blend,1,20,-1.129,-0.844,-0.103,-0.122,0.512,0.5,0.80,99.635
...,...,...,...,...,...,...,...,...,...,...,...,...
85,Japan,mr_ret_5 + mr_ret_20,weekly,2,0.720,-0.325,0.066,0.204,0.133,0.0,0.75,21.182
86,Japan,mr_ret_5 + mr_ret_20,weekly,5,0.614,-0.333,0.056,0.167,0.133,0.0,0.75,21.182
87,Japan,mr_ret_5 + mr_ret_20,weekly,10,0.439,-0.346,0.038,0.110,0.133,0.0,0.75,21.182
88,Japan,mr_ret_5 + mr_ret_20,weekly,20,0.088,-0.370,0.004,0.010,0.133,0.0,0.75,21.182


In [5]:
from src.analysis.deployment import select_best_per_group

best_rebalance_by_cost = select_best_per_group(
    rebalance_tc_results_df,
    group_cols=["Market", "Cost bps"],
    score_col="Sharpe",
    maximize=True,
)

best_rebalance_by_cost

,Market,Cost bps,Signal Combo,Rebalance Frequency,Sharpe,MDD,CAGR,Calmar,Mean Turnover,Median Turnover,P95 Turnover,% Trading Days
0,Brazil,0,mom_blend + mr_lowvol_blend,weekly,2.045,-0.250,0.217,0.869,0.147,0.00,0.833,20.980
1,Brazil,2,mom_blend + mr_lowvol_blend,weekly,1.970,-0.252,0.208,0.826,0.147,0.00,0.833,20.980
2,Brazil,5,mom_blend + mr_lowvol_blend,weekly,1.857,-0.255,0.194,0.763,0.147,0.00,0.833,20.980
3,Brazil,10,mom_blend + mr_lowvol_blend,10,1.667,-0.290,0.176,0.605,0.084,0.00,0.857,10.039
4,Brazil,20,mom_blend + mr_lowvol_blend,10,1.451,-0.306,0.151,0.493,0.084,0.00,0.857,10.039
5,Brazil,50,mom_blend + mr_lowvol_blend,10,0.804,-0.409,0.080,0.195,0.084,0.00,0.857,10.039
6,India,0,mr_ret_10 + low_vol_20 + mr_lowvol_blend,1,2.118,-0.220,0.235,1.071,0.285,0.25,0.500,97.476
7,India,2,mr_ret_10 + low_vol_20 + mr_lowvol_blend,1,1.978,-0.238,0.218,0.914,0.285,0.25,0.500,97.476
8,India,5,mr_ret_10 + low_vol_20 + mr_lowvol_blend,1,1.767,-0.266,0.192,0.720,0.285,0.25,0.500,97.476
9,India,10,mr_ret_10 + low_vol_20 + mr_lowvol_blend,2,1.530,-0.269,0.163,0.607,0.191,0.00,0.556,49.816


The combined rebalance frequency and transaction cost analysis demonstrates that the optimal implementation of the deployment strategies depends on both market characteristics and trading costs. For India, daily rebalancing produced the highest gross performance under negligible transaction costs; however, the optimal frequency shifted to a two-day rebalance once costs reached approximately 5 bps, reflecting the trade-off between alpha capture and turnover. Brazil exhibited significantly more persistent signals, with weekly rebalancing remaining optimal across low-cost environments before transitioning to a ten-day schedule under higher transaction costs. Japan displayed intermediate behaviour, favouring two-day rebalancing at low transaction costs and five-day rebalancing as costs increased. These findings show that rebalance frequency should be considered a deployable strategy parameter rather than a fixed design choice, with implementation decisions tailored to both market dynamics and expected execution costs.

In [6]:
from pathlib import Path

results_dir = Path("../results")
results_dir.mkdir(exist_ok=True)

rebalance_tc_results_df.to_csv(
    results_dir / "rebalance_transaction_cost_results.csv",
    index=False,
)

print("Saved rebalance_transaction_cost_results.csv")

Saved rebalance_transaction_cost_results.csv


In [7]:
best_rebalance_by_cost.to_csv(
    results_dir / "best_rebalance_by_transaction_cost.csv",
    index=False,
)

print("Saved best_rebalance_by_transaction_cost.csv")

Saved best_rebalance_by_transaction_cost.csv


In [8]:
from src.analysis.deployment import pivot_metric_table

deployment_rebalance_table = pivot_metric_table(
    best_rebalance_by_cost,
    index="Cost bps",
    columns="Market",
    value_col="Rebalance Frequency",
)

deployment_rebalance_table

Market,Brazil,India,Japan
Cost bps,,,
0,weekly,1,2
2,weekly,1,2
5,weekly,1,2
10,10,2,5
20,10,2,5
50,10,5,5


In [9]:
deployment_rebalance_table.to_csv(
    results_dir / "deployment_rebalance_summary.csv"
)

print("Saved deployment_rebalance_summary.csv")

Saved deployment_rebalance_summary.csv
